# Fast R-CNN — Fast Region-based Convolutional Network (2015)

_Papers · Computer Vision_

**Run the convolutions ONCE for the whole image, then pool each region from the shared feature map and train classifier + box regressor together in one pass.**

---

This notebook builds Fast R-CNN's core ideas from scratch, one piece at a time: RoI pooling, the multi-task loss, and the multi-task vs single-task ablation. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build Fast R-CNN's two key pieces, then test the idea. In four steps: (1) RoI max pooling by hand, (2) the multi-task loss on a worked example, (3) checking our pooling against torchvision's oracle, and (4) the multi-task vs single-task ablation from §5.1.

### Step 1 — RoI-pool a feature map by hand

Fast R-CNN runs the convolutions once, then pools each region from the shared feature map. **RoI max pooling** splits the region's `h × w` window into a fixed `H × W` grid and takes the max in each sub-window — so any region becomes a fixed-size output the fully-connected head can accept.

Here we pool the whole 4×4 map down to a 2×2 grid and verify the four max values by eye.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import roi_pool   # the ORACLE we check our hand-written pooling against

torch.manual_seed(0)

# A tiny 4x4 single-channel feature map to pool.
fmap = torch.tensor([[1., 3, 2, 4],
                     [5., 6, 1, 2],
                     [7., 2, 8, 3],
                     [0., 9, 4, 1]])

def roi_pool_manual(fm, y0, x0, h, w, H, W):
    # Split the h x w window into an H x W grid; max each sub-window (one channel).
    out = torch.zeros(H, W)
    for i in range(H):
        for j in range(W):
            ys = y0 + (i * h) // H
            ye = y0 + ((i + 1) * h) // H
            xs = x0 + (j * w) // W
            xe = x0 + ((j + 1) * w) // W
            out[i, j] = fm[ys:ye, xs:xe].max()
    return out

# Pool the whole 4x4 map (h=w=4) into a 2x2 grid (H=W=2).
pooled = roi_pool_manual(fmap, 0, 0, 4, 4, 2, 2)
print("worked RoI-pool 2x2:\n", pooled.tolist())   # [[6, 4], [9, 8]]

### Step 2 — The multi-task loss `L = Lcls + λ·[u≥1]·Lloc`

Fast R-CNN trains the classifier and the box regressor **together** with one loss (Eqn. 1). The classification term is `-log p_u` (Eqn. 1), and the localization term is a **smooth-L1** sum over the four box coordinates (Eqn. 2–3).

The `[u≥1]` switch zeroes the box loss for background regions — there is no box to regress for "nothing". Here `u=1` (a real class), so both terms count.

In [ ]:
def smooth_l1(x):
    a = x.abs()
    return torch.where(a < 1.0, 0.5 * x * x, a - 0.5)   # Eqn. 3

# Classification term: -log of the predicted probability for the true class u.
pu = torch.tensor(0.6)
Lcls = -torch.log(pu)                                   # -log p_u  = 0.5108

# Localization term: smooth-L1 over the four box-coordinate errors (t - v).
t = torch.tensor([0.2, -0.1, 0.5, -2.0])
v = torch.zeros(4)
Lloc = smooth_l1(t - v).sum()                           # Eqn. 2    = 1.65

# Combine: box loss only counts for a real class (u >= 1), scaled by lambda.
u = 1
lam = 1.0
indicator = 1.0 if u >= 1 else 0.0
L = Lcls + lam * indicator * Lloc                       # Eqn. 1
print("Lcls=%.4f  Lloc=%.4f  L=%.4f" % (Lcls, Lloc, L))  # 0.5108  1.6500  2.1608

### Step 3 — Check our pooling against torchvision's oracle

To trust the hand-written pooling, we compare it to `torchvision.ops.roi_pool`. torchvision expects features `(N,C,Hf,Wf)` and boxes as `[batch_idx, x1, y1, x2, y2]` in feature coordinates, where `x2`/`y2` are the **inclusive** max cell index.

The 4×4 map spans indices 0..3, so the whole-map RoI is `[0, 0, 3, 3]` — this makes torchvision's binning line up with our `h/H` split, and the two pooled grids should match exactly.

In [ ]:
# torchvision wants the feature map as (N, C, Hf, Wf).
feat = fmap.view(1, 1, 4, 4)

# Whole-map RoI in [batch_idx, x1, y1, x2, y2] form, with INCLUSIVE x2/y2.
boxes = torch.tensor([[0., 0., 0., 3., 3.]])

ref = roi_pool(feat, boxes, output_size=(2, 2), spatial_scale=1.0)[0, 0]
mine = roi_pool_manual(fmap, 0, 0, 4, 4, 2, 2)

print("torchvision roi_pool:\n", ref.tolist())
print("allclose(mine, torchvision.ops.roi_pool):", torch.allclose(mine, ref))
# allclose == True  -> our RoI pooling IS torchvision's.

### Step 4 — Multi-task vs single-task (the §5.1 ablation)

Does training the box regressor alongside the classifier help classification? We build toy "regions": **noisy** pooled features plus a **clean** per-class characteristic box. Because the box task pulls the shared trunk toward class structure, it should help classification on a held-out test set.

We train the same head two ways — `λ=1` (multi-task) and `λ=0` (classification only) — changing exactly one thing, and compare test accuracy. The box loss is ignored for background (`u=0`) via the `[u≥1]` switch.

In [ ]:
# Toy "regions": noisy features, plus a clean per-class characteristic box.
g = torch.Generator().manual_seed(0)
Din, K, Ntr, Nte = 12, 4, 180, 400        # K real classes (labels 1..K); 0 = background

centers = torch.randn(K + 1, Din, generator=g)             # class feature centers
box_centers = torch.randn(K + 1, 4, generator=g) * 4.0     # each class's characteristic box

def make(n):                                               # one batch of toy regions
    y = torch.randint(0, K + 1, (n,), generator=g)
    X = centers[y] + torch.randn(n, Din, generator=g) * 2.0    # VERY noisy features
    V = box_centers[y] + torch.randn(n, 4, generator=g) * 0.2  # clean per-class box target
    objectness = (y >= 1).float().unsqueeze(1)                 # 1 for real class, 0 for background
    return X, y, objectness, V

Xtr, ytr, obj_tr, Vtr = make(Ntr)
Xte, yte, _, _ = make(Nte)                 # held-out test set for the accuracy we report

class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(Din, 24), nn.ReLU())   # SHARED trunk
        self.cls = nn.Linear(24, K + 1)            # softmax over K+1 classes
        self.box = nn.Linear(24, 4)                # bbox regressor t^u
    def forward(self, x):
        f = self.trunk(x)
        return self.cls(f), self.box(f)

def train(lmbda, steps=150, lr=0.05):
    torch.manual_seed(1)
    net = Head()
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    ce = nn.CrossEntropyLoss()
    for _ in range(steps):
        opt.zero_grad()
        logits, t = net(Xtr)
        Lcls = ce(logits, ytr)                                    # Eqn. 1 classification term
        Lloc = (obj_tr * smooth_l1(t - Vtr)).sum(1).mean()        # Eqn. 2, switched by [u>=1]
        loss = Lcls + lmbda * Lloc                                # Eqn. 1
        loss.backward()
        opt.step()
    with torch.no_grad():
        return (net(Xte)[0].argmax(1) == yte).float().mean().item()

acc_multi = train(lmbda=1.0)     # multi-task: classification + box loss
acc_single = train(lmbda=0.0)    # single-task ABLATION: classification only
print("test classification acc  single-task (lambda=0):  %.4f" % acc_single)  # ~0.5250
print("test classification acc  multi-task  (lambda=1):  %.4f" % acc_multi)   # ~0.5525
print("multi-task helps classification:", acc_multi >= acc_single)            # True
# Multi-task >= single-task here, reproducing 5.1's qualitative effect.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does adding the smooth-L1 box-regression loss (multi-task) improve classification accuracy vs training the classifier alone (single-task)? (§5.1)_

### Step 1 — Rebuild the toy regions and head

This panel reproduces §5.1's effect as a standalone, self-contained run. We regenerate the same noisy features with a clean per-class box, and define the shared-trunk head with a classification branch and a box branch.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
torch.manual_seed(0)

g = torch.Generator().manual_seed(0)
Din, K, Ntr, Nte = 12, 4, 180, 400

centers = torch.randn(K + 1, Din, generator=g)
box_centers = torch.randn(K + 1, 4, generator=g) * 4.0

def make(n):
    y = torch.randint(0, K + 1, (n,), generator=g)
    X = centers[y] + torch.randn(n, Din, generator=g) * 2.0    # noisy features
    V = box_centers[y] + torch.randn(n, 4, generator=g) * 0.2  # clean per-class box
    objectness = (y >= 1).float().unsqueeze(1)
    return X, y, objectness, V

Xtr, ytr, obj_tr, Vtr = make(Ntr)
Xte, yte, _, _ = make(Nte)

def smooth_l1(x):
    a = x.abs()
    return torch.where(a < 1.0, 0.5 * x * x, a - 0.5)

class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(Din, 24), nn.ReLU())
        self.cls = nn.Linear(24, K + 1)
        self.box = nn.Linear(24, 4)
    def forward(self, x):
        f = self.trunk(x)
        return self.cls(f), self.box(f)

### Step 2 — Train both ways and compare accuracy

We run the same training twice: `λ=0` (classification only) and `λ=1` (classification + smooth-L1 box loss). With everything else held identical, any accuracy gap is attributable to the box loss shaping the shared trunk — reproducing §5.1's qualitative result on our toy data.

In [ ]:
def train(lmbda, steps=150, lr=0.05):
    torch.manual_seed(1)
    net = Head()
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    ce = nn.CrossEntropyLoss()
    for _ in range(steps):
        opt.zero_grad()
        logits, t = net(Xtr)
        loss = ce(logits, ytr) + lmbda * (obj_tr * smooth_l1(t - Vtr)).sum(1).mean()
        loss.backward()
        opt.step()
    with torch.no_grad():
        return (net(Xte)[0].argmax(1) == yte).float().mean().item()

print("single-task (lambda=0):", round(train(0.0), 4))   # 0.525
print("multi-task  (lambda=1):", round(train(1.0), 4))   # 0.5525

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation (&sect;5.1). You have a working detection head trained multi-task
            (classification + smooth-L1 box loss). Set $\lambda=0$ so the box loss is switched off
            (single-task: classification only), retrain, and measure classification accuracy. What do
            you expect, and what does the result demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change exactly one thing: set $\lambda = 0$ in the multi-task loss (the box head still exists, but its loss no longer affects the shared layers). Keep data, depth, optimizer, and seed identical. — _An honest ablation isolates the multi-task signal: any difference in classification accuracy is attributable to the box loss being present or absent._
- Retrain and compare classification accuracy of the multi-task ($\lambda=1$) vs single-task ($\lambda=0$) runs. — _The paper&rsquo;s &sect;5.1 reports multi-task training improves classification over training it alone, because the box-regression signal shapes the shared features._
- Conclude whether the shared representation benefits from learning both tasks at once. — _If multi-task wins, the auxiliary box loss acts as a useful regularizer/extra signal on the shared trunk._

**Answer:** The multi-task classifier should match or beat the single-task one: in the paper&rsquo;s
                 &sect;5.1 ("Does multi-task training help?") multi-task training improves classification
                 accuracy by roughly $+0.8$ to $+1.1$ mAP. Since the two runs differ only in $\lambda$
                 ($1$ vs $0$), this isolates the box-regression loss as the cause &mdash; the extra task shapes
                 the shared features and helps classification. The CODEVIZ panel reproduces this contrast on
                 toy data (our numbers, not the paper&rsquo;s).

</details>

**Problem 2.** A region proposal maps to a $6\times8$ rectangle on the feature map, and you must RoI-pool it to
            $H\times W = 3\times2$. What is each sub-window&rsquo;s size, and how many input cells does each
            output value max over?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Sub-window height $= h/H = 6/3 = 2$; width $= w/W = 8/2 = 4$. — _&sect;2.1: the $h\times w$ window is split into an $H\times W$ grid of sub-windows of approximate size $h/H\times w/W$._
- Each output cell maxes over a $2\times4$ block of feature-map cells. — _RoI max pooling takes the maximum value inside each sub-window into the corresponding output cell._
- The output is a fixed $3\times2$ grid regardless of the region&rsquo;s original size. — _That fixed size is exactly what lets the downstream fully-connected head accept any region._

**Answer:** Each sub-window is $h/H \times w/W = 2\times4$ feature-map cells, so each of the
                 $3\times2 = 6$ output values is the max over 8 cells. The output is a fixed $3\times2$
                 grid no matter the region&rsquo;s input size &mdash; that is the point of RoI pooling.

</details>

**Problem 3.** In the worked example the worst coordinate error was $-2.0$ and contributed $1.5$ to $L_\text{loc}$.
            What would plain squared error $\tfrac12 x^2$ have contributed for that coordinate, and why does
            the paper prefer smooth-L1?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Squared error: $\tfrac12(-2.0)^2 = \tfrac12\cdot 4 = 2.0$. — _Squared error grows quadratically, so a large error is penalized much more heavily than the linear tail._
- Smooth-L1 (Eqn. 3, $|x|\ge1$ branch): $|{-2.0}| - 0.5 = 1.5$. — _Once $|x|\ge 1$ smooth-L1 is linear, so it caps how fast the loss &mdash; and its gradient &mdash; grows._
- Compare gradients: squared error gives $|x|=2$; smooth-L1 gives $\operatorname{sign}(x)=\pm1$. — _The capped gradient means one badly-off coordinate cannot dominate the update &mdash; no exploding gradients._

**Answer:** Squared error would contribute $\tfrac12(2)^2 = 2.0$ (vs smooth-L1&rsquo;s $1.5$), and its
                 gradient would be $2$ (vs smooth-L1&rsquo;s capped $\pm1$). The paper prefers smooth-L1 because
                 it is "less sensitive to outliers than the $L_2$ loss" &mdash; the bounded tail gradient
                 prevents the exploding gradients that an unbounded $L_2$ box loss can cause.

</details>